# 27. Deep Q-Network (DQN)
**Use case:** Oil & Gas — Natural gas storage: daily Inject / Hold / Withdraw decisions

A midstream operator leases a 100,000 MMBtu storage field and must decide each day whether to inject gas, hold, or withdraw and sell. Prices are cheap in summer and expensive in winter, but the size of the winter premium and short-term shocks change every year. Schedulers today use simple buy-below/sell-above rules that leave money on the table and sometimes never trigger. Data available: daily hub prices, forward curves, storage levels, injection/withdrawal limits and fuel costs. Success means a policy that beats the rule-based baseline out-of-sample, with fewer loss-making years, inside all physical and contractual limits.

> 💡 Think of a new gas scheduler who keeps a notebook of 'if the price is X and the tank is Y full, injecting today tends to pay Z by year end'. At first the notebook is guesses; after many simulated years it becomes a reliable rulebook. DQN is that notebook, written by a neural network.

Legend in code: `# 🐍 PY:` Python lesson · `# 🧠 ML:` ML lesson

In [ ]:
# Run once (Colab already has most of these)
!pip -q install numpy pandas scikit-learn scipy xgboost torch

## EASY: A tiny neural Q-function on five storage transitions
Train a 2-8-3 neural network by hand in numpy so it learns which of Inject/Hold/Withdraw is worth most in two storage situations.

| 5W + H | |
|---|---|
| **Who** | A storage analyst new to reinforcement learning. |
| **What** | Five hand-typed transitions (state, action, reward, next state) from a gas field, with price and fill level as the state. |
| **When** | Before building a full simulator: to see what a Q-network is and how the Bellman target trains it. |
| **Where** | A single numpy script, no deep-learning library, so every gradient is visible. |
| **Why** | DQN is just this idea scaled up; understanding the target and the backprop removes the mystery. |
| **How** | Forward pass, Bellman target r + gamma x max Q(next), squared error on the chosen action, manual backprop, repeat 300 epochs. |

**🧠 ML concepts:** Q-function approximated by a neural network, Bellman target, Temporal-difference (TD) error, Backpropagation through one hidden layer, Greedy policy from Q-values  
**🐍 Python concepts:** Lists of tuples holding numpy arrays, Functions returning multiple values, The @ matrix-multiplication operator, Tuple unpacking in for-loops, In-place array updates with -=, np.outer for gradient matrices

In [ ]:
# A tiny neural Q-function for gas storage: Inject / Hold / Withdraw (Easy, plain numpy)
import numpy as np

np.random.seed(42)
ACTIONS = ["Inject", "Hold", "Withdraw"]
# 🐍 PY: each transition is a tuple (state, action, reward, next_state)
# state = [scaled gas price, storage fill fraction]
transitions = [
    # reward for Inject = winter forward value of the gas minus today's purchase cost (mark-to-market)
    (np.array([0.20, 0.10]), 0, +3.0, np.array([0.25, 0.30])),  # cheap summer gas -> inject
    (np.array([0.20, 0.10]), 1, -0.2, np.array([0.25, 0.10])),  # hold in summer -> only a storage fee
    (np.array([0.90, 0.80]), 2, +9.0, np.array([0.85, 0.60])),  # pricey winter gas -> withdraw (we earn)
    (np.array([0.50, 0.50]), 1, -0.2, np.array([0.55, 0.50])),  # hold -> small storage fee
    (np.array([0.20, 0.10]), 2, -5.0, np.array([0.25, 0.10])),  # withdraw when empty-ish and cheap -> penalty
]
gamma, lr = 0.5, 0.02  # modest discount keeps this tiny bootstrapped problem stable

# 🧠 ML: Q-network = one hidden layer: 2 inputs -> 8 hidden -> 3 outputs (one Q-value per action)
W1 = np.random.randn(2, 8) * 0.5
b1 = np.zeros(8)
W2 = np.random.randn(8, 3) * 0.5
b2 = np.zeros(3)

# 🐍 PY: a function returning two values as a tuple
def forward(s):
    h = np.maximum(0, s @ W1 + b1)  # 🧠 ML: ReLU hidden layer
    return h, h @ W2 + b2           # 🐍 PY: '@' is matrix multiplication

for epoch in range(301):
    loss = 0.0
    # 🐍 PY: tuple unpacking straight inside the for-loop header
    for s, a, r, s2 in transitions:
        h, q = forward(s)
        _, q_next = forward(s2)
        # 🧠 ML: Bellman target = reward now + discounted best future value
        target = r + gamma * q_next.max()
        err = q[a] - target
        loss += err ** 2
        # 🧠 ML: gradient only flows through the chosen action's output (backprop by hand)
        dq = np.zeros(3)
        dq[a] = err
        dh = (W2 @ dq) * (h > 0)
        # 🐍 PY: np.outer builds the weight-gradient matrix; '-=' updates arrays in place
        W2 -= lr * np.outer(h, dq)
        b2 -= lr * dq
        W1 -= lr * np.outer(s, dh)
        b1 -= lr * dh
    if epoch % 100 == 0:
        print(f"epoch {epoch:3d}  squared TD error = {loss:.3f}")

# 🧠 ML: the greedy policy simply picks the action with the highest predicted Q
for label, s in [("Summer, cheap, nearly empty", [0.2, 0.1]), ("Winter, pricey, fairly full", [0.9, 0.8])]:
    _, q = forward(np.array(s))
    print(f"{label:30s} Q={np.round(q, 2)} -> {ACTIONS[int(q.argmax())]}")

**Try this next:**
- Set gamma to 0.9 and lr to 0.05: does training still converge, or do Q-values blow up (a classic DQN instability)?
- Add a transition where withdrawing in summer earns +1 and see how the summer Q-values change.
- Increase the hidden layer to 32 units and compare the final Q-values.

## MEDIUM: DQN with replay buffer on a seasonal gas-price year
Train a PyTorch DQN with a replay buffer on a simulated storage year and show its month-by-month inject/withdraw pattern.

| 5W + H | |
|---|---|
| **Who** | A commercial analytics team prototyping an RL scheduler for one storage field. |
| **What** | A simulator with 120 three-day decision steps per year, seasonal prices plus noise, 100k MMBtu capacity and 8k MMBtu max flow per step. |
| **When** | Prototype phase, before investing in a richer simulator with fuel costs and price shocks. |
| **Where** | A Python script with PyTorch on CPU; trains in seconds. |
| **Why** | Shows whether a learning agent discovers the fill-in-summer, sell-in-winter pattern on its own. |
| **How** | Epsilon-greedy actions, experiences stored in a deque, random mini-batches trained toward r + gamma x max Q(next) with Smooth L1 loss. |

**🧠 ML concepts:** Environment / agent loop, Experience replay, Epsilon-greedy exploration, Mark-to-market reward shaping, Huber (Smooth L1) loss for Q-learning, Greedy evaluation after training  
**🐍 Python concepts:** Classes with methods and class constants, collections.deque with maxlen, zip(*batch) to transpose a list of tuples, torch.no_grad context manager, Tensor gather and max, f-string column alignment

In [ ]:
# DQN with a replay buffer: daily Inject / Hold / Withdraw decisions for a gas storage field (Medium)
import random
from collections import deque
import numpy as np
import torch
import torch.nn as nn

torch.set_num_threads(1)
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
ACTIONS = ["Inject", "Hold", "Withdraw"]


class GasStorageEnv:
    """One storage year from 1 April in 120 decision steps of 3 days. Cheap summer gas, dear winter gas."""
    CAPACITY, RATE, FEE, STEPS = 100.0, 8.0, 0.01, 120  # thousand MMBtu, per step, $/unit/step, steps/yr

    def reset(self):
        self.day, self.storage, self.cash = 0, 0.0, 0.0
        # 🧠 ML: seasonal price curve: ~$2 around July, ~$4 around January, random noise each year
        t = np.arange(self.STEPS + 1)
        self.prices = 3 - np.cos(2 * np.pi * (t - 30) / self.STEPS) + np.random.normal(0, 0.15, self.STEPS + 1)
        return self._state()

    def _state(self):
        # 🐍 PY: np.array of 4 floats - scaled price, fill level, and the calendar as sin/cos
        ang = 2 * np.pi * self.day / self.STEPS
        return np.array([self.prices[self.day] - 3, self.storage / self.CAPACITY, np.sin(ang), np.cos(ang)],
                        dtype=np.float32)

    def step(self, action):
        p = self.prices[self.day]
        # 🐍 PY: min/max clamp the volume so we never overfill or go below empty
        vol = 0.0
        if action == 0:
            vol = min(self.RATE, self.CAPACITY - self.storage)
        elif action == 2:
            vol = -min(self.RATE, self.storage)
        self.storage += vol
        self.day += 1
        # 🧠 ML: mark-to-market reward = gain in value of gas held (cash + inventory at today's price) - fee
        # 🧠 ML: a dense signal like this is far easier to learn from than 'cash only when we finally sell'
        reward = self.storage * (self.prices[self.day] - p) - self.FEE * self.storage
        self.cash += -vol * p
        return self._state(), reward / 10.0, self.day >= self.STEPS  # reward scaled for stable learning


# 🧠 ML: the Q-network maps a 4-number state to 3 Q-values (expected future profit of each action)
qnet = nn.Sequential(nn.Linear(4, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 3))
opt = torch.optim.Adam(qnet.parameters(), lr=1e-3)
# 🐍 PY: deque(maxlen=...) is a fixed-size queue - oldest memories drop off automatically
replay = deque(maxlen=20000)
env, gamma, eps, batch = GasStorageEnv(), 0.97, 0.10, 64


def act(state, eps):
    # 🧠 ML: epsilon-greedy - mostly exploit the best Q, sometimes try a random action to explore
    if random.random() < eps:
        return random.randrange(3)
    with torch.no_grad():
        return int(qnet(torch.from_numpy(state)).argmax())


for ep in range(1, 61):
    s, done, total = env.reset(), False, 0.0
    while not done:
        a = act(s, eps)
        s2, r, done = env.step(a)
        # 🐍 PY: append a tuple to the buffer; later we sample random mini-batches from it
        replay.append((s, a, r, s2, done))
        s, total = s2, total + r
        if len(replay) >= 500 and env.day % 2 == 0:
            # 🧠 ML: random sampling breaks the day-to-day correlation that destabilises training
            S, A, R, S2, D = zip(*random.sample(replay, batch))
            S, S2 = torch.tensor(np.array(S)), torch.tensor(np.array(S2))
            A, R, D = torch.tensor(A), torch.tensor(R, dtype=torch.float32), torch.tensor(D, dtype=torch.float32)
            q = qnet(S).gather(1, A.unsqueeze(1)).squeeze(1)
            with torch.no_grad():
                target = R + gamma * (1 - D) * qnet(S2).max(1).values
            loss = nn.functional.smooth_l1_loss(q, target)
            opt.zero_grad()
            loss.backward()
            opt.step()
    if ep % 10 == 0:
        print(f"episode {ep:2d}  mark-to-market P&L=${total * 10:7.1f}k  buffer={len(replay)}")

# 🧠 ML: evaluate greedily (eps=0) on a fresh price year and show the monthly pattern
s, done, log = env.reset(), False, []
while not done:
    a = act(s, 0.0)
    log.append((env.day, env.prices[env.day], a, env.storage))
    s, r, done = env.step(a)
MONTHS = ["Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec", "Jan", "Feb", "Mar"]
print("\nMonth  avg_price  most_common_action  end_storage")
for m in range(12):
    # 🐍 PY: slicing a list of tuples - 10 steps of 3 days = one month
    chunk = log[m * 10:(m + 1) * 10]
    acts = [c[2] for c in chunk]
    print(f"{MONTHS[m]:>5s}  ${np.mean([c[1] for c in chunk]):8.2f}  {ACTIONS[max(set(acts), key=acts.count)]:>18s}"
          f"  {env.storage if m == 11 else log[(m + 1) * 10][3]:10.0f}")
value = env.cash + env.storage * env.prices[-1]
print(f"\nBusiness read: year-end value (cash + gas left) = ${value:.0f}k on a 100k MMBtu field.")
print("A good policy fills in the cheap summer months and withdraws into the winter price peak.")

**Try this next:**
- Set eps to 0.3 during training: does the policy get better or worse?
- Change the reward to cash only (-vol * p) and see how much harder learning becomes.
- Double FEE to 0.02 and check whether the agent starts filling later in the summer.

## HARD: Double DQN with target network vs tuned buy-low-sell-high rule
Train a Double DQN with target network and epsilon decay on a richer storage simulator, then compare it with a tuned rule over 200 unseen price years.

| 5W + H | |
|---|---|
| **Who** | A storage optimisation team preparing evidence for the trading risk committee. |
| **What** | Random winter premium each year, persistent AR(1) price shocks, injection and withdrawal fuel costs, 120 decisions a year. |
| **When** | Before any pilot: the agent must beat the incumbent rule out-of-sample with fewer loss-making years. |
| **Where** | An offline training job; weights saved with torch.save for the policy service. |
| **Why** | Rules with fixed thresholds miss mild winters and price shocks; a learned policy can adapt to time of year, inventory and price together. |
| **How** | Target network synced every 200 updates, Double-DQN targets, gradient clipping, epsilon decay 1.0 to 0.05, thresholds of the rule tuned on training years. |

**🧠 ML concepts:** Target network for stable targets, Double DQN to reduce over-estimation, Epsilon decay schedule, Gradient clipping, Out-of-sample evaluation over many episodes, Tuning the baseline fairly, Policy maps for explainability  
**🐍 Python concepts:** Per-object RandomState for reproducible, separate data streams, Closures (functions that return functions), Passing functions as arguments, map(np.array, zip(*...)) for batching, state_dict save/load, Percentiles with numpy

In [ ]:
# DQN with target network + epsilon decay vs a tuned buy-low-sell-high rule for gas storage (Hard)
import os
import random
import tempfile
import time
from collections import deque
import numpy as np
import torch
import torch.nn as nn

torch.set_num_threads(1)
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
ACTIONS = ["Inject", "Hold", "Withdraw"]


class GasStorageEnv:
    """A storage year from 1 April in 120 steps of 3 days, with fuel costs and price regime shocks."""
    CAPACITY, RATE, FEE, STEPS = 100.0, 8.0, 0.01, 120
    INJ_COST, WDR_COST = 0.04, 0.02  # $/MMBtu compressor fuel + variable O&M

    def __init__(self, seed):
        # 🐍 PY: each env owns its own RandomState so train and test price paths never overlap
        self.rng = np.random.RandomState(seed)

    def reset(self):
        self.t, self.storage, self.cash = 0, 0.0, 0.0
        steps = np.arange(self.STEPS + 1)
        # 🧠 ML: richer prices = random winter premium + AR(1) noise (shocks persist for a while)
        amp = self.rng.uniform(0.6, 1.4)
        noise = np.zeros(self.STEPS + 1)
        for i in range(1, self.STEPS + 1):
            noise[i] = 0.85 * noise[i - 1] + self.rng.normal(0, 0.12)
        self.prices = np.maximum(0.5, 3 - amp * np.cos(2 * np.pi * (steps - 30) / self.STEPS) + noise)
        return self.state()

    def state(self):
        ang = 2 * np.pi * self.t / self.STEPS
        return np.array([self.prices[self.t] - 3, self.storage / self.CAPACITY, np.sin(ang), np.cos(ang),
                         1 - self.t / self.STEPS], dtype=np.float32)

    def step(self, a):
        p = self.prices[self.t]
        vol = min(self.RATE, self.CAPACITY - self.storage) if a == 0 else (-min(self.RATE, self.storage) if a == 2 else 0)
        cost = self.INJ_COST * max(vol, 0) + self.WDR_COST * max(-vol, 0) + self.FEE * self.storage
        self.storage += vol
        self.cash += -vol * p - cost
        self.t += 1
        # 🧠 ML: mark-to-market reward: sums (telescopes) to final cash + inventory value
        reward = self.storage * (self.prices[self.t] - p) - cost
        return self.state(), reward / 10.0, self.t >= self.STEPS

    def value(self):
        return self.cash + self.storage * self.prices[self.t]


def make_qnet():
    return nn.Sequential(nn.Linear(5, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 3))


def train_dqn(episodes=120, gamma=0.97, lr=1e-3, batch=64, sync_every=200):
    env, qnet, target = GasStorageEnv(SEED), make_qnet(), make_qnet()
    # 🧠 ML: target network = a frozen copy used to compute targets; stops the 'chasing your own tail' problem
    target.load_state_dict(qnet.state_dict())
    opt = torch.optim.Adam(qnet.parameters(), lr=lr)
    replay, updates = deque(maxlen=30000), 0
    for ep in range(episodes):
        # 🧠 ML: epsilon decays from 1.0 (pure exploration) to 0.05 over the first 70% of training
        eps = max(0.05, 1.0 - ep / (0.7 * episodes))
        s, done, ep_r = env.reset(), False, 0.0
        while not done:
            if random.random() < eps:
                a = random.randrange(3)
            else:
                with torch.no_grad():
                    a = int(qnet(torch.from_numpy(s)).argmax())
            s2, r, done = env.step(a)
            replay.append((s, a, r, s2, float(done)))
            s, ep_r = s2, ep_r + r
            if len(replay) < 1000 or env.t % 2:
                continue
            S, A, R, S2, D = map(np.array, zip(*random.sample(replay, batch)))
            S, S2 = torch.from_numpy(S), torch.from_numpy(S2)
            A, R, D = torch.from_numpy(A), torch.tensor(R, dtype=torch.float32), torch.tensor(D, dtype=torch.float32)
            q = qnet(S).gather(1, A.unsqueeze(1)).squeeze(1)
            with torch.no_grad():
                # 🧠 ML: Double-DQN target: online net CHOOSES the next action, target net SCORES it
                a_next = qnet(S2).argmax(1, keepdim=True)
                y = R + gamma * (1 - D) * target(S2).gather(1, a_next).squeeze(1)
            loss = nn.functional.smooth_l1_loss(q, y)
            opt.zero_grad()
            loss.backward()
            # 🧠 ML: gradient clipping guards against rare exploding updates from price spikes
            nn.utils.clip_grad_norm_(qnet.parameters(), 5.0)
            opt.step()
            updates += 1
            if updates % sync_every == 0:
                target.load_state_dict(qnet.state_dict())
        if (ep + 1) % 30 == 0:
            print(f"  episode {ep + 1:3d}  eps={eps:.2f}  train value=${ep_r * 10:6.1f}k  updates={updates}")
    return qnet


def evaluate(policy, n_years=200, seed=999):
    # 🐍 PY: 'policy' is any function state->action, so the DQN and the rule share one evaluator
    env, vals = GasStorageEnv(seed), []
    for _ in range(n_years):
        s, done = env.reset(), False
        while not done:
            s, _, done = env.step(policy(s, env))
        vals.append(env.value())
    return np.array(vals)


def rule_policy(buy_below, sell_above):
    # 🐍 PY: a closure - the inner function 'remembers' the two thresholds
    def policy(s, env):
        p = env.prices[env.t]
        return 0 if p < buy_below else (2 if p > sell_above else 1)
    return policy


def main():
    t0 = time.time()
    # 🧠 ML: give the baseline a fair chance - tune its thresholds on TRAINING years, not test years
    grid = [(b, s) for b in (2.2, 2.5, 2.8) for s in (3.2, 3.5, 3.8)]
    scores = {g: evaluate(rule_policy(*g), n_years=60, seed=SEED).mean() for g in grid}
    best_rule = max(scores, key=scores.get)
    print(f"Best rule on training years: buy < ${best_rule[0]}, sell > ${best_rule[1]} "
          f"(avg ${scores[best_rule]:.1f}k)")

    print("Training Double DQN with target network...")
    qnet = train_dqn()
    print(f"Training + tuning time: {time.time() - t0:.1f}s")

    def dqn_policy(s, env):
        with torch.no_grad():
            return int(qnet(torch.from_numpy(s)).argmax())

    # 🧠 ML: evaluate on 200 unseen price years - the only honest way to compare policies
    res = {"Do nothing": evaluate(lambda s, e: 1), "Rule (tuned)": evaluate(rule_policy(*best_rule)),
           "DQN": evaluate(dqn_policy)}
    print("\nOut-of-sample results over 200 price years ($k per year):")
    print(f"{'policy':14s} {'mean':>7s} {'p5':>7s} {'p95':>7s} {'loss yrs':>9s}")
    for name, v in res.items():
        print(f"{name:14s} {v.mean():7.1f} {np.percentile(v, 5):7.1f} {np.percentile(v, 95):7.1f} "
              f"{(v < 0).mean():9.1%}")
    diff = res["DQN"] - res["Rule (tuned)"]
    print(f"DQN beats rule in {(diff > 0).mean():.0%} of years; mean uplift ${diff.mean():.1f}k")

    # 🧠 ML: explainability - what does the network do at key (season, fill level) points?
    print("\nPolicy map (rows = fill level, cols = month index, price at seasonal average):")
    env = GasStorageEnv(1)
    env.reset()
    print("fill  " + " ".join(f"{m:>2d}" for m in range(1, 13)))
    for fill in (0.0, 0.5, 1.0):
        row = []
        for m in range(12):
            t = m * 10 + 5
            ang = 2 * np.pi * t / 120
            p = 3 - np.cos(2 * np.pi * (t - 30) / 120)
            s = torch.tensor([p - 3, fill, np.sin(ang), np.cos(ang), 1 - t / 120], dtype=torch.float32)
            row.append(" " + ACTIONS[int(qnet(s).argmax())][0])
        print(f"{fill:4.1f} " + " ".join(row))
    print("(I=Inject H=Hold W=Withdraw; month 1 = April)")

    out = os.path.join(tempfile.mkdtemp(), "gas_storage_dqn.pt")
    # 🐍 PY: state_dict() is a plain dict of tensors - the portable way to save torch weights
    torch.save(qnet.state_dict(), out)
    print(f"\nSaved Q-network weights to {out}")
    print("MODEL-RISK NOTE: treat the agent as decision SUPPORT. Pipeline nominations, FERC/tariff limits,")
    print("ratchets and credit limits must be hard constraints outside the model; re-validate when the")
    print("forward curve regime changes (SR 11-7 style ongoing monitoring for trading models).")


if __name__ == "__main__":
    main()

**Try this next:**
- Set sync_every to 1 (no real target network) and compare the out-of-sample mean.
- Raise INJ_COST to 0.10 and see how the policy map and uplift over the rule change.
- Train for 60 episodes instead of 120: how much performance do you lose?